# build_2khz_tensor_dict

Builds the `segfilt_2khz_emg_tensor_dict.pkl` file needed for the A10/A11/A12 Meta model ablations.

**Pipeline:**
1. Load raw segmented EMG CSVs (16 channels × ~6000 samples at 2000 Hz)
2. BPF 20–450 Hz + mean subtraction
3. Per-gesture std normalisation (std ≈ 1 across all channels per trial)
4. Survey trial-length distribution → choose TARGET_LEN_SAMPLES
5. Resample all trials to TARGET_LEN_SAMPLES at 2000 Hz (preserves 2 kHz spectral content)
6. Pack into tensor_dict and save as `.pkl`

**Run order:** Execute cells top-to-bottom. Cell 4 (survey) prints the length distribution — read its output before running Cell 5 onwards. Intermediate JSON saves mean you can resume from Cell 3 if the kernel dies.

**After this notebook:** `scp` the saved `.pkl` to the cluster, then update `EMG_2KHZ_PKL_PATH` and `EMG_2KHZ_SEQ_LEN` in `A10_A11_A12_meta_pretrained.py`.

**Prevent sleep before running Cell 2** — this takes ~50 minutes for the CSV load.

## 0. Imports

In [1]:
import json
import pickle
import time
from pathlib import Path

import numpy as np
import torch
from scipy.signal import resample as scipy_resample

from shared_processing import (
    load_segraw_data,
    apply_filter_to_nested_dict,
    normalize_gestures_by_std_any_channels,
)

## 1. Config — set your paths here

In [ ]:
# ── UPDATE THESE TO YOUR MACHINE ──────────────────────────────────────────────
RAW_DATA_DIR = r"C:\Users\kdmen\Box\Yamagami Lab\Data\2024_UIST_dataset\upload\segmented_raw_data"
SAVE_DIR     = r"C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project"
# ──────────────────────────────────────────────────────────────────────────────

# Intermediate saves — if the kernel dies you can reload from these
# instead of re-running the 50-minute CSV load
RAW_DICT_SAVE_PATH  = SAVE_DIR + r"\segraw0_2khz_EMG.json"
PPD_DICT_SAVE_PATH  = SAVE_DIR + r"\ppdsegraw1_2khz_EMG.json"

# Final output — this is what you scp to the cluster
PKL_SAVE_PATH = SAVE_DIR + r"\segfilt_2khz_emg_tensor_dict.pkl"

FS = 2000  # Hz — do not change

# Set this AFTER running Cell 4 (survey).
# Rule: use the p10 value from the survey output, rounded down to nearest 100.
# 2000 samples = 1.0 s at 2 kHz is a reasonable starting guess.
TARGET_LEN_SAMPLES = 4300

PIDS_IMPAIRED   = ['P102','P103','P104','P105','P106','P107','P108','P109','P110','P111',
                   'P112','P114','P115','P116','P118','P119','P121','P122','P123','P124',
                   'P125','P126','P127','P128','P131','P132']
PIDS_UNIMPAIRED = ['P004','P005','P006','P008','P010','P011']
PIDS_ALL        = PIDS_IMPAIRED + PIDS_UNIMPAIRED

# 0-indexed gesture class labels — must match your existing tensor_dict exactly.
# Verify against your ablation_config.py or existing pkl before proceeding.
GESTURE_TO_CLASS = {
    'close':         0,
    'delete':        1,
    'duplicate':     2,
    'move':          3,
    'open':          4,
    'pan':           5,
    'rotate':        6,
    'select-single': 7,
    'zoom-in':       8,
    'zoom-out':      9,
}

print("Config loaded.")
print(f"  RAW_DATA_DIR : {RAW_DATA_DIR}")
print(f"  PKL_SAVE_PATH: {PKL_SAVE_PATH}")
print(f"  Participants : {len(PIDS_ALL)}")
print(f"  Gesture classes: {GESTURE_TO_CLASS}")

Config loaded.
  RAW_DATA_DIR : C:\Users\kdmen\Box\Yamagami Lab\Data\2024_UIST_dataset\upload\segmented_raw_data
  PKL_SAVE_PATH: C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\segfilt_2khz_emg_tensor_dict.pkl
  Participants : 32
  Gesture classes: {'close': 0, 'delete': 1, 'duplicate': 2, 'move': 3, 'open': 4, 'pan': 5, 'rotate': 6, 'select-single': 7, 'zoom-in': 8, 'zoom-out': 9}


## 2. Load raw EMG CSVs (~50 minutes)

The intermediate save at the end means if anything goes wrong in later cells,
you can reload from `segraw0_2khz_EMG.json` instead of waiting 50 minutes again.

In [3]:
print("Loading raw EMG CSVs...")
t0 = time.time()

nested_dict = load_segraw_data(
    pIDs             = PIDS_ALL,
    data_dir_path    = RAW_DATA_DIR,
    modalities       = ["E"],
    expt_types       = ["experimenter-defined"],
    num_emg_channels = 16,
)

print(f"Done in {time.time()-t0:.1f}s")
print(f"Participants loaded: {list(nested_dict.keys())}")

# Sanity check shape
sample_pid     = list(nested_dict.keys())[0]
sample_gesture = list(nested_dict[sample_pid].keys())[0]
sample_trial   = list(nested_dict[sample_pid][sample_gesture].keys())[0]
sample_data    = nested_dict[sample_pid][sample_gesture][sample_trial]["EMG"]
print(f"\nSample: {sample_pid} / {sample_gesture} / trial {sample_trial}")
print(f"  n_channels  : {len(sample_data)}   (expected 16)")
print(f"  n_timepoints: {len(sample_data[0])}   (expected ~4000–7000)")

Loading raw EMG CSVs...
P102
P103
P104
P105
P106
P107
P108
P109
P110
P111
P112
P114
P115
P116
P118
P119
P121
P122
P123
P124
P125
P126
P127
P128
P131
P132
P004
P005
P006
P008
P010
P011
Done in 209.0s
Participants loaded: ['P102', 'P103', 'P104', 'P105', 'P106', 'P107', 'P108', 'P109', 'P110', 'P111', 'P112', 'P114', 'P115', 'P116', 'P118', 'P119', 'P121', 'P122', 'P123', 'P124', 'P125', 'P126', 'P127', 'P128', 'P131', 'P132', 'P004', 'P005', 'P006', 'P008', 'P010', 'P011']

Sample: P102 / pan / trial 1
  n_channels  : 16   (expected 16)
  n_timepoints: 6380   (expected ~4000–7000)


In [4]:
# Optional intermediate save — skip if you're short on disk space
print("Saving raw dict (intermediate checkpoint)...")
with open(RAW_DICT_SAVE_PATH, 'w') as f:
    json.dump(nested_dict, f)
print(f"Saved → {RAW_DICT_SAVE_PATH}")

Saving raw dict (intermediate checkpoint)...
Saved → C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\segraw0_2khz_EMG.json


## 3. BPF 20–450 Hz + mean subtraction + per-gesture std normalisation

In [5]:
print("Applying BPF 20–450 Hz + mean subtraction...")
t0 = time.time()

filt_dict = apply_filter_to_nested_dict(
    nested_dict,
    normalization_method = "MEANSUBTRACTION",
    already_BPFd         = False,
)

print(f"Done in {time.time()-t0:.1f}s")

Applying BPF 20–450 Hz + mean subtraction...
Done in 162.4s


In [6]:
print("Per-gesture std normalisation...")
t0 = time.time()

ppd_dict = normalize_gestures_by_std_any_channels(filt_dict)

print(f"Done in {time.time()-t0:.1f}s")

# Sanity check: std across flattened channels should be ~1
sample_data_ppd = ppd_dict[sample_pid][sample_gesture][sample_trial]["EMG"]
flat = [v for ch in sample_data_ppd for v in ch]
print(f"Spot-check std (should be ~1.0): {np.std(flat):.4f}")

Per-gesture std normalisation...
Done in 77.8s
Spot-check std (should be ~1.0): 1.0000


In [7]:
# Optional intermediate save
print("Saving preprocessed dict (intermediate checkpoint)...")
with open(PPD_DICT_SAVE_PATH, 'w') as f:
    json.dump(ppd_dict, f)
print(f"Saved → {PPD_DICT_SAVE_PATH}")

Saving preprocessed dict (intermediate checkpoint)...
Saved → C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\ppdsegraw1_2khz_EMG.json


## 4. Survey trial-length distribution

**Read the output of this cell before proceeding.**
Set `TARGET_LEN_SAMPLES` in Cell 1 (config) based on the p10 value printed here,
then re-run Cell 1 before continuing to Cell 5.

In [8]:
lengths = []
for pid in ppd_dict:
    for gesture in ppd_dict[pid]:
        for trial in ppd_dict[pid][gesture]:
            ch0 = ppd_dict[pid][gesture][trial]["EMG"][0]
            lengths.append(len(ch0))

lengths = np.array(lengths)

print("=== Trial Length Distribution (samples at 2000 Hz) ===")
print(f"  N trials : {len(lengths)}")
print(f"  Min      : {lengths.min():,}  ({lengths.min()/FS*1000:.0f} ms)")
print(f"  p5       : {int(np.percentile(lengths,  5)):,}  ({np.percentile(lengths,  5)/FS*1000:.0f} ms)")
print(f"  p10      : {int(np.percentile(lengths, 10)):,}  ({np.percentile(lengths, 10)/FS*1000:.0f} ms)")
print(f"  p25      : {int(np.percentile(lengths, 25)):,}  ({np.percentile(lengths, 25)/FS*1000:.0f} ms)")
print(f"  Median   : {int(np.percentile(lengths, 50)):,}  ({np.percentile(lengths, 50)/FS*1000:.0f} ms)")
print(f"  p75      : {int(np.percentile(lengths, 75)):,}  ({np.percentile(lengths, 75)/FS*1000:.0f} ms)")
print(f"  Max      : {lengths.max():,}  ({lengths.max()/FS*1000:.0f} ms)")
print(f"  Std      : {lengths.std():.0f} samples  ({lengths.std()/FS*1000:.0f} ms)")

suggested = int(np.percentile(lengths, 10)) // 100 * 100
print(f"\n→ Suggested TARGET_LEN_SAMPLES = {suggested}")
print(f"  (p10 rounded down to nearest 100 = {suggested/FS*1000:.0f} ms)")
print()
print("If this looks reasonable, update TARGET_LEN_SAMPLES in Cell 1 and re-run it,")
print("then continue to Cell 5.")

=== Trial Length Distribution (samples at 2000 Hz) ===
  N trials : 3200
  Min      : 1,580  (790 ms)
  p5       : 3,620  (1810 ms)
  p10      : 4,340  (2170 ms)
  p25      : 5,079  (2540 ms)
  Median   : 6,519  (3260 ms)
  p75      : 7,960  (3980 ms)
  Max      : 24,999  (12500 ms)
  Std      : 2450 samples  (1225 ms)

→ Suggested TARGET_LEN_SAMPLES = 4300
  (p10 rounded down to nearest 100 = 2150 ms)

If this looks reasonable, update TARGET_LEN_SAMPLES in Cell 1 and re-run it,
then continue to Cell 5.


## 5. Resample all trials and build tensor_dict

Make sure you have updated `TARGET_LEN_SAMPLES` in Cell 1 and re-run it before continuing.

In [ ]:
def resample_trial(emg_channels: list, target_len: int) -> np.ndarray:
    """
    Resample one gesture trial to exactly target_len samples.

    Args:
        emg_channels : list of 16 lists, each of length T_original.
        target_len   : desired output length in samples.

    Returns:
        np.ndarray of shape (target_len, 16) — channel-last, matching
        your tensor_dict's (seq_len, num_channels) convention.
    """
    arr = np.array(emg_channels, dtype=np.float32)  # (16, T_orig)

    if arr.shape[1] == target_len:
        return arr.T  # (target_len, 16) — no resampling needed

    # scipy_resample along axis=1: (16, T_orig) → (16, target_len)
    arr_resampled = scipy_resample(arr, target_len, axis=1)

    return arr_resampled.T.astype(np.float32)  # (target_len, 16)


print(f"Building tensor_dict with TARGET_LEN_SAMPLES={TARGET_LEN_SAMPLES}...")
t0 = time.time()

tensor_dict = {}
pid_list = sorted(ppd_dict.keys())

for pid_idx, pid in enumerate(pid_list):
    tensor_dict[pid] = {}
    print(f"  {pid} ({pid_idx+1}/{len(pid_list)})...", end=" ", flush=True)

    for gesture_name, gesture_class in GESTURE_TO_CLASS.items():
        if gesture_name not in ppd_dict[pid]:
            print(f"\n  WARNING: '{gesture_name}' not found for {pid}, skipping.")
            continue

        trial_keys  = sorted(ppd_dict[pid][gesture_name].keys())
        trial_arrays = []

        for trial_key in trial_keys:
            emg_channels = ppd_dict[pid][gesture_name][trial_key]["EMG"]
            arr = resample_trial(emg_channels, TARGET_LEN_SAMPLES)  # (T, 16)
            trial_arrays.append(arr)

        emg_tensor = torch.from_numpy(
            np.stack(trial_arrays, axis=0)  # (num_trials, T, 16)
        ).float()

        assert emg_tensor.shape == (len(trial_keys), TARGET_LEN_SAMPLES, 16), (
            f"Shape error for {pid}/{gesture_name}: {emg_tensor.shape}"
        )

        tensor_dict[pid][gesture_class] = {
            "emg":         emg_tensor,
            "imu":         None,
            "demo":        torch.zeros(12),  # placeholder; not used by Meta model
            "gest_ID":     gesture_class,
            "enc_gest_ID": gesture_class,
            "enc_pid":     pid_idx,
            "rep_indices": [int(k) if str(k).isdigit() else i + 1
                            for i, k in enumerate(trial_keys)],
        }

    print("done")

print(f"\nFinished in {time.time()-t0:.1f}s")

## 6. Verify

In [ ]:
print("=== Verification ===")
pids = list(tensor_dict.keys())
print(f"Participants : {len(pids)}")
print(f"Gesture classes for {pids[0]}: {sorted(tensor_dict[pids[0]].keys())}")

# Shape check — every entry must be (num_trials, TARGET_LEN_SAMPLES, 16)
errors = []
for pid in pids:
    for cls in tensor_dict[pid]:
        emg = tensor_dict[pid][cls]["emg"]
        if emg.shape[1] != TARGET_LEN_SAMPLES or emg.shape[2] != 16:
            errors.append(f"{pid}/class{cls}: {emg.shape}")
        if torch.isnan(emg).any() or torch.isinf(emg).any():
            errors.append(f"{pid}/class{cls}: contains NaN or Inf")

if errors:
    print(f"ERRORS ({len(errors)}):")
    for e in errors:
        print(f"  {e}")
else:
    print("All shape and NaN/Inf checks passed.")

# Value sanity: per-trial std should be ~1 after normalisation
sample_cls = sorted(tensor_dict[pids[0]].keys())[0]
sample_emg = tensor_dict[pids[0]][sample_cls]["emg"]  # (N, T, 16)
per_trial_stds = sample_emg.std(dim=[1, 2])
print(f"\nSample {pids[0]}/class{sample_cls} — shape: {sample_emg.shape}")
print(f"Per-trial std: mean={per_trial_stds.mean():.3f}, "
      f"min={per_trial_stds.min():.3f}, max={per_trial_stds.max():.3f}")
print("(should be ~1.0)")

## 7. Save

In [ ]:
print(f"Saving to {PKL_SAVE_PATH} ...")
t0 = time.time()

with open(PKL_SAVE_PATH, "wb") as f:
    pickle.dump({"data": tensor_dict}, f, protocol=4)

import os
size_mb = os.path.getsize(PKL_SAVE_PATH) / 1e6
print(f"Done in {time.time()-t0:.1f}s")
print(f"File size: {size_mb:.0f} MB")
print(f"Saved → {PKL_SAVE_PATH}")
print()
print("Next steps:")
print(f"  1. scp this file to your cluster:")
print(f"     scp '{PKL_SAVE_PATH}' km82@<cluster>:/rhf/allocations/my13/div-emg/dataset/")
print(f"  2. In A10_A11_A12_meta_pretrained.py, set:")
print(f"     EMG_2KHZ_PKL_PATH  = Path('/rhf/allocations/my13/div-emg/dataset/segfilt_2khz_emg_tensor_dict.pkl')")
print(f"     EMG_2KHZ_SEQ_LEN   = {TARGET_LEN_SAMPLES}")